In [1]:
import random
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, f1_score, average_precision_score

import torch
from torch.nn import BCEWithLogitsLoss
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EvalPrediction,
    EarlyStoppingCallback
)

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold

torch.manual_seed(4262)
np.random.seed(4262)

## Load Dataset

In [2]:
train_df = pd.read_csv('../data/model_training_data.csv')
train_df.head()

,title,selftext,Assignee,feas_pa,feas_ka,feas_cr,feas_sr
0,"Accepted a Job, Relocated, and Then Got My Off...","In other words, they filled the role internall...",Overlap,1.0,1.0,3.0,1.0
1,I Am a Fraud and ChatGPT Is My Brain,literally everything. uni? fake. career pat...,Overlap,4.0,4.0,4.0,1.0
2,Wow interviews suck more now,"This makes you look bad"" and I replied ""if you...",Overlap,1.0,1.0,1.0,2.0
3,Think I'm about to turn Netflix down. Am I crazy?,Did all 7 interviews down and got an offer. 5...,Overlap,1.0,1.0,1.0,1.0
4,"The bar is absolutely, insanely high.",Interviewed at a unicorn tech company for inte...,Overlap,1.0,1.0,1.0,1.0


In [3]:
# Convert to binary labels (0 for <=3, 1 for >3)
train_df['feas_pa'] = (train_df['feas_pa'] > 3).astype(int)
train_df['feas_ka'] = (train_df['feas_ka'] > 3).astype(int)
train_df['feas_cr'] = (train_df['feas_cr'] > 3).astype(int)
train_df['feas_sr'] = (train_df['feas_sr'] > 3).astype(int)
train_df.head()

,title,selftext,Assignee,feas_pa,feas_ka,feas_cr,feas_sr
0,"Accepted a Job, Relocated, and Then Got My Off...","In other words, they filled the role internall...",Overlap,0,0,0,0
1,I Am a Fraud and ChatGPT Is My Brain,literally everything. uni? fake. career pat...,Overlap,1,1,1,0
2,Wow interviews suck more now,"This makes you look bad"" and I replied ""if you...",Overlap,0,0,0,0
3,Think I'm about to turn Netflix down. Am I crazy?,Did all 7 interviews down and got an offer. 5...,Overlap,0,0,0,0
4,"The bar is absolutely, insanely high.",Interviewed at a unicorn tech company for inte...,Overlap,0,0,0,0


In [4]:
# Print class distribution percentages
print("Class distribution for feas_pa:")
print(train_df['feas_pa'].value_counts(normalize=True) * 100)
print("\nClass distribution for feas_ka:")
print(train_df['feas_ka'].value_counts(normalize=True) * 100)
print("\nClass distribution for feas_cr:")
print(train_df['feas_cr'].value_counts(normalize=True) * 100)
print("\nClass distribution for feas_sr:")
print(train_df['feas_sr'].value_counts(normalize=True) * 100)

Class distribution for feas_pa:
feas_pa
0    80.913978
1    19.086022
Name: proportion, dtype: float64

Class distribution for feas_ka:
feas_ka
0    93.27957
1     6.72043
Name: proportion, dtype: float64

Class distribution for feas_cr:
feas_cr
0    89.247312
1    10.752688
Name: proportion, dtype: float64

Class distribution for feas_sr:
feas_sr
0    98.924731
1     1.075269
Name: proportion, dtype: float64


In [5]:
test_df = pd.read_csv('../data/form_responses.csv')
test_df.head()

,timestamp,consent,demo_gender,demo_area_of_study,demo_honours_class,demo_job_search_status,demo_num_prior_interns,demo_financial_urgency,feas_pa1,feas_pa2,...,feas_sr3,feas_sr4,bhv_apps_4_wks,bhv_itvs_4_wks,bhv_job_update_check_daily,ac2,bhv_app_duration,bhv_app_status_check_daily,bhv_app_avoidance_weekly,bhv_job_search_exp
0,2/24/2026 8:47:57,I agree,Male,Computing and Data,Honours (Highest Distinction),Actively applying,1-2,5,5,2,...,1,2,31+,2 - 3,1 - 2,0,<10 min,1 - 2,1 - 2,Not knowing what the companies want. Sometimes...
1,2/24/2026 11:51:05,I agree,Female,Computing and Data,Honours (Distinction),Actively applying,1-2,3,5,4,...,4,2,6 - 15,1,1 - 2,0,10 - 30 min,0,3 - 5,I dl being on LinkedIn sometimes it’s a luck g...
2,2/24/2026 11:55:42,I agree,Female,Design & Engineering,Honours (Highest Distinction),Actively applying,1-2,2,4,2,...,4,2,16 - 30,0,3 - 5,0,10 - 30 min,1 - 2,1 - 2,1. Not making good progress - not trying hard ...
3,2/24/2026 12:07:31,I agree,Male,Design & Engineering,Honours (Distinction),Actively applying,1-2,5,2,2,...,4,4,0,1,3 - 5,0,10 - 30 min,1 - 2,0,1. Social circle: I feel worried when I hear o...
4,2/24/2026 12:08:09,I agree,Female,Computing and Data,Honours (Distinction),Secured but still actively applying,3 and above,2,4,4,...,2,4,16 - 30,1,1 - 2,0,10 - 30 min,1 - 2,3 - 5,Seeing people get to big companies feels a lit...


In [6]:
# Compute average score for 4 dimensions
test_df['feas_pa'] = test_df[['feas_pa1', 'feas_pa2', 'feas_pa3', 'feas_pa4', 'feas_pa5']].mean(axis=1)
test_df['feas_ka'] = test_df[['feas_ka1', 'feas_ka2', 'feas_ka3', 'feas_ka4']].mean(axis=1)
test_df['feas_cr'] = test_df[['feas_cr1', 'feas_cr2', 'feas_cr3', 'feas_cr4']].mean(axis=1)
test_df['feas_sr'] = test_df[['feas_sr1', 'feas_sr2', 'feas_sr3', 'feas_sr4']].mean(axis=1)

In [7]:
# Convert to binary labels (0 for <=3, 1 for >3)
test_df['feas_pa'] = (test_df['feas_pa'] > 3).astype(int)
test_df['feas_ka'] = (test_df['feas_ka'] > 3).astype(int)
test_df['feas_cr'] = (test_df['feas_cr'] > 3).astype(int)
test_df['feas_sr'] = (test_df['feas_sr'] > 3).astype(int)
test_df.head()

,timestamp,consent,demo_gender,demo_area_of_study,demo_honours_class,demo_job_search_status,demo_num_prior_interns,demo_financial_urgency,feas_pa1,feas_pa2,...,bhv_job_update_check_daily,ac2,bhv_app_duration,bhv_app_status_check_daily,bhv_app_avoidance_weekly,bhv_job_search_exp,feas_pa,feas_ka,feas_cr,feas_sr
0,2/24/2026 8:47:57,I agree,Male,Computing and Data,Honours (Highest Distinction),Actively applying,1-2,5,5,2,...,1 - 2,0,<10 min,1 - 2,1 - 2,Not knowing what the companies want. Sometimes...,0,0,0,0
1,2/24/2026 11:51:05,I agree,Female,Computing and Data,Honours (Distinction),Actively applying,1-2,3,5,4,...,1 - 2,0,10 - 30 min,0,3 - 5,I dl being on LinkedIn sometimes it’s a luck g...,1,1,0,0
2,2/24/2026 11:55:42,I agree,Female,Design & Engineering,Honours (Highest Distinction),Actively applying,1-2,2,4,2,...,3 - 5,0,10 - 30 min,1 - 2,1 - 2,1. Not making good progress - not trying hard ...,0,1,1,1
3,2/24/2026 12:07:31,I agree,Male,Design & Engineering,Honours (Distinction),Actively applying,1-2,5,2,2,...,3 - 5,0,10 - 30 min,1 - 2,0,1. Social circle: I feel worried when I hear o...,0,1,0,1
4,2/24/2026 12:08:09,I agree,Female,Computing and Data,Honours (Distinction),Secured but still actively applying,3 and above,2,4,4,...,1 - 2,0,10 - 30 min,1 - 2,3 - 5,Seeing people get to big companies feels a lit...,1,1,1,1


In [8]:
TARGET_COLS = ['feas_pa', 'feas_ka', 'feas_cr', 'feas_sr']

def prepare_text(df):
    return np.array(df['title'].fillna('') + ' ' + df['selftext'].fillna(''))

X_full = prepare_text(train_df)
y_full = np.array(train_df[TARGET_COLS])

X_test = np.array(test_df['bhv_job_search_exp'].fillna(''))
y_test = np.array(test_df[TARGET_COLS])

# 2. Initialize the multi-label stratifier
# We use n_splits=1 because we just want a single train/val split
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# 3. Generate the split
for train_index, val_index in msss.split(X_full, y_full):
    X_train, X_val = X_full[train_index], X_full[val_index]
    y_train, y_val = y_full[train_index], y_full[val_index]

print("Shapes of datasets:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Shapes of datasets:
X_train: (297,)
y_train: (297, 4)
X_val: (75,)
y_val: (75, 4)
X_test: (55,)
y_test: (55, 4)


In [9]:
def check_distributions(y_array, split_name, target_cols):
    """Prints the count and percentage of positive samples for each label."""
    percentages = np.mean(y_array, axis=0) * 100
    counts = np.sum(y_array, axis=0)
    total_samples = len(y_array)
    
    print(f"=== {split_name} Set (Total Samples: {total_samples}) ===")
    
    dist_df = pd.DataFrame({
        'Positive Count': counts,
        'Positive %': percentages
    }, index=target_cols)
    
    dist_df['Positive %'] = dist_df['Positive %'].map('{:.2f}%'.format)
    
    display(dist_df)
    print("\n")

check_distributions(y_train, "Train", TARGET_COLS)
check_distributions(y_val, "Validation", TARGET_COLS)
check_distributions(y_test, "Test", TARGET_COLS)

=== Train Set (Total Samples: 297) ===


,Positive Count,Positive %
feas_pa,57,19.19%
feas_ka,20,6.73%
feas_cr,32,10.77%
feas_sr,3,1.01%




=== Validation Set (Total Samples: 75) ===


,Positive Count,Positive %
feas_pa,14,18.67%
feas_ka,5,6.67%
feas_cr,8,10.67%
feas_sr,1,1.33%




=== Test Set (Total Samples: 55) ===


,Positive Count,Positive %
feas_pa,29,52.73%
feas_ka,37,67.27%
feas_cr,32,58.18%
feas_sr,26,47.27%


## 1. TF-IDF + Logistic Regression

In [10]:
# TF-IDF is inside pipeline → fitted per CV fold
# Logistic Regression wrapped with OneVsRest

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('clf', OneVsRestClassifier(
        LogisticRegression(
            solver='liblinear',
            class_weight='balanced',
            max_iter=1000
        )
    ))
])

In [11]:
# Important: we cannot directly do multi-label stratified CV easily with sklearn
# Practical workaround: 1) Tune using one representative label (e.g. feas_pa); 2) Then retrain full model on all labels

param_grid = {
    'tfidf__max_features': [5000, 10000],
    'tfidf__ngram_range': [(1,1), (1,2)],
    'clf__estimator__C': [0.1, 1, 5]
}

# 1. Use MultilabelStratifiedKFold to properly stratify across ALL 4 labels
cv = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1_macro', # 2. Evaluate the macro-averaged F1 score across all labels
    verbose=2,
    n_jobs=-1
)

# 3. Fit on the full multi-label array (y_train) instead of just the sliced y_strat
grid.fit(X_train, y_train) 

print("Best params:", grid.best_params_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.1s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.1s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.1s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.0s


/Users/wy/Documents/DSA4262-Anxiety/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/wy/Documents/DSA4262-Anxiety/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/wy/Documents/DSA4262-Anxiety/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.


[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.1s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 2); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 2); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 2); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=1000

/Users/wy/Documents/DSA4262-Anxiety/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/wy/Documents/DSA4262-Anxiety/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/wy/Documents/DSA4262-Anxiety/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.


[CV] END clf__estimator__C=5, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=5, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.0s
[CV] END clf__estimator__C=5, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=5, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.0s
[CV] END clf__estimator__C=5, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=5, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.0s
[CV] END clf__estimator__C=5, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.1s
[CV] END clf__estimator__C=5, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.1s
[CV] END clf__estimator__C=5, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=5, tfidf__max_features=10000, tfidf__ngram_range=(1

/Users/wy/Documents/DSA4262-Anxiety/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/wy/Documents/DSA4262-Anxiety/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [12]:
# Train final model on all labels with best params
best_pipeline = grid.best_estimator_

best_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [13]:
# Prediction
y_prob_val = best_pipeline.predict_proba(X_val)
y_prob_test = best_pipeline.predict_proba(X_test)

In [14]:
# Threshold tuning (on validation set to prevent data leakage)
def tune_threshold(y_true, y_prob):
    if np.sum(y_true) == 0:
        return 0.5  # default threshold

    best_t, best_f1 = 0.5, 0
    for t in np.linspace(0.05, 0.95, 50):
        preds = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t


thresholds = []

for i in range(len(TARGET_COLS)):
    t = tune_threshold(y_val[:, i], y_prob_val[:, i])
    thresholds.append(t)

print("Tuned thresholds:", thresholds)

y_pred_tuned = (y_prob_test >= thresholds).astype(int)

Tuned thresholds: [np.float64(0.4908163265306122), np.float64(0.4908163265306122), np.float64(0.4724489795918367), np.float64(0.3255102040816326)]


In [15]:
# Evaluation (per label)
for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (TFIDF + LR)=====")
    print(classification_report(y_test[:, i], y_pred_tuned[:, i], digits=4, zero_division=0))


===== feas_pa (TFIDF + LR)=====
              precision    recall  f1-score   support

           0     0.4839    0.5769    0.5263        26
           1     0.5417    0.4483    0.4906        29

    accuracy                         0.5091        55
   macro avg     0.5128    0.5126    0.5084        55
weighted avg     0.5143    0.5091    0.5075        55


===== feas_ka (TFIDF + LR)=====
              precision    recall  f1-score   support

           0     0.3396    1.0000    0.5070        18
           1     1.0000    0.0541    0.1026        37

    accuracy                         0.3636        55
   macro avg     0.6698    0.5270    0.3048        55
weighted avg     0.7839    0.3636    0.2349        55


===== feas_cr (TFIDF + LR)=====
              precision    recall  f1-score   support

           0     0.6364    0.3043    0.4118        23
           1     0.6364    0.8750    0.7368        32

    accuracy                         0.6364        55
   macro avg     0.6364    0.

In [16]:
# Evaluation (overall)
macro_f1 = f1_score(y_test, y_pred_tuned, average='macro')
micro_f1 = f1_score(y_test, y_pred_tuned, average='micro')

print("Macro F1:", macro_f1)
print("Micro F1:", micro_f1)

Macro F1: 0.4929868885512712
Micro F1: 0.5542168674698795


## 2. Sentence Embeddings + LightGBM

In [17]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

X_train_emb = embedder.encode(X_train, batch_size=16, show_progress_bar=True)
X_val_emb = embedder.encode(X_val, batch_size=16, show_progress_bar=True)
X_test_emb = embedder.encode(X_test, batch_size=16, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [18]:
# Compute imbalance weight
def compute_scale_pos_weight(y):
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)
    return neg / (pos + 1e-6)

# Random parameter sampler
param_dist = {
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 400, 800],
    "min_child_samples": [5, 10, 20],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0]
}

def sample_params(param_dist, n_iter=20):
    keys = list(param_dist.keys())
    sampled = []
    
    for _ in range(n_iter):
        params = {k: random.choice(param_dist[k]) for k in keys}
        sampled.append(params)
    
    return sampled


import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning
)

In [19]:
best_params_per_label = []

# 1. Determine the safe number of splits based on the globally rarest label
# We need at least 2 splits for CV. Using the rarest class ensures the stratifier doesn't crash.
min_positives_overall = int(y_train.sum(axis=0).min())
n_splits = min(5, max(2, min_positives_overall))

print(f"Using {n_splits}-fold Multi-label Stratified CV for all models")

# 2. Create the unified folds ONCE for all labels based on the full y_train matrix
mskf = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
unified_folds = list(mskf.split(X_train_emb, y_train))

for label_idx, col in enumerate(TARGET_COLS):
    print(f"\n==============================")
    print(f"Tuning for {col}")
    print(f"==============================")

    y_label = y_train[:, label_idx]

    if y_label.sum() < 2:
        print(f"Skipping {col} (too few samples for CV)")
        best_params_per_label.append(None)
        continue
        
    scale_pos_weight = compute_scale_pos_weight(y_label)
    param_list = sample_params(param_dist, n_iter=20)

    best_score = -1
    best_params = None

    for params in param_list:
        fold_scores = []

        # 3. Iterate through the pre-computed unified folds
        for train_idx, val_idx in unified_folds:
            X_fold_train, X_fold_val = X_train_emb[train_idx], X_train_emb[val_idx]
            y_fold_train, y_fold_val = y_label[train_idx], y_label[val_idx]

            model = lgb.LGBMClassifier(
                **params,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            )

            model.fit(
                X_fold_train, y_fold_train,
                eval_set=[(X_fold_val, y_fold_val)],
                eval_metric="average_precision",
                callbacks=[lgb.early_stopping(30, verbose=False)]
            )

            y_prob = model.predict_proba(X_fold_val)[:, 1]
            score = average_precision_score(y_fold_val, y_prob)
            fold_scores.append(score)

        mean_score = np.mean(fold_scores)

        if mean_score > best_score:
            best_score = mean_score
            best_params = params

    print(f"Best PR-AUC: {best_score:.4f}")
    print(f"Best Params: {best_params}")

    best_params_per_label.append(best_params)

Using 3-fold Multi-label Stratified CV for all models

Tuning for feas_pa
Best PR-AUC: 0.5174
Best Params: {'num_leaves': 31, 'learning_rate': 0.01, 'n_estimators': 800, 'min_child_samples': 20, 'subsample': 1.0, 'colsample_bytree': 0.7}

Tuning for feas_ka
Best PR-AUC: 0.4788
Best Params: {'num_leaves': 15, 'learning_rate': 0.01, 'n_estimators': 200, 'min_child_samples': 10, 'subsample': 1.0, 'colsample_bytree': 1.0}

Tuning for feas_cr
Best PR-AUC: 0.2165
Best Params: {'num_leaves': 63, 'learning_rate': 0.05, 'n_estimators': 200, 'min_child_samples': 10, 'subsample': 0.7, 'colsample_bytree': 0.9}

Tuning for feas_sr
Best PR-AUC: 0.0837
Best Params: {'num_leaves': 63, 'learning_rate': 0.05, 'n_estimators': 800, 'min_child_samples': 20, 'subsample': 0.9, 'colsample_bytree': 0.7}


In [20]:
# Train final model
models = []

for label_idx, col in enumerate(TARGET_COLS):
    print(f"\nTraining final model for {col}")

    y_label = y_train[:, label_idx]
    scale_pos_weight = compute_scale_pos_weight(y_label)

    model = lgb.LGBMClassifier(
        **best_params_per_label[label_idx],
        scale_pos_weight=scale_pos_weight,
        random_state=42
    )

    model.fit(X_train_emb, y_label)
    models.append(model)


Training final model for feas_pa

Training final model for feas_ka

Training final model for feas_cr

Training final model for feas_sr


In [21]:
# Prediction
num_models = len(models)

y_prob_val = np.zeros((len(y_val), num_models))
y_prob_test = np.zeros((len(y_test), num_models))

for i, model in enumerate(models):
    y_prob_val[:, i] = model.predict_proba(X_val_emb)[:, 1]
    y_prob_test[:, i] = model.predict_proba(X_test_emb)[:, 1]

In [22]:
# Threshold tuning on validation set to prevent data leakage
thresholds = []

for i in range(len(TARGET_COLS)):
    t = tune_threshold(y_val[:, i], y_prob_val[:, i])
    thresholds.append(t)

print("Best thresholds:", thresholds)

y_pred_tuned_lgbm = np.zeros_like(y_prob_test, dtype=int)
for i in range(len(TARGET_COLS)):
    y_pred_tuned_lgbm[:, i] = (y_prob_test[:, i] >= thresholds[i]).astype(int)

Best thresholds: [np.float64(0.2520408163265306), np.float64(0.06836734693877551), np.float64(0.06836734693877551), 0.5]


In [23]:
# Evaluation (per label)
for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (Embedding + LightGBM) =====")
    print(classification_report(y_test[:, i], y_pred_tuned_lgbm[:, i], digits=4, zero_division=0))


===== feas_pa (Embedding + LightGBM) =====
              precision    recall  f1-score   support

           0     0.4773    0.8077    0.6000        26
           1     0.5455    0.2069    0.3000        29

    accuracy                         0.4909        55
   macro avg     0.5114    0.5073    0.4500        55
weighted avg     0.5132    0.4909    0.4418        55


===== feas_ka (Embedding + LightGBM) =====
              precision    recall  f1-score   support

           0     0.3429    0.6667    0.4528        18
           1     0.7000    0.3784    0.4912        37

    accuracy                         0.4727        55
   macro avg     0.5214    0.5225    0.4720        55
weighted avg     0.5831    0.4727    0.4787        55


===== feas_cr (Embedding + LightGBM) =====
              precision    recall  f1-score   support

           0     0.4468    0.9130    0.6000        23
           1     0.7500    0.1875    0.3000        32

    accuracy                         0.4909       

In [24]:
# Evaluation (overall)
macro_f1 = f1_score(y_test, y_pred_tuned, average='macro')
micro_f1 = f1_score(y_test, y_pred_tuned, average='micro')

print("Macro F1:", macro_f1)
print("Micro F1:", micro_f1)

Macro F1: 0.4929868885512712
Micro F1: 0.5542168674698795


## 3. Fine-tuned DistilBERT for Multi-label Classification

### PyTorch Dataset Definition

In [25]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class FEASDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        # Multi-label classification requires float tensors
        labels = torch.tensor(self.labels[idx], dtype=torch.float)
        
        encoding = self.tokenizer(
            text, 
            truncation=True, 
            padding='max_length', 
            max_length=self.max_length, 
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': labels
        }

# Create the training dataset
train_dataset = FEASDataset(X_train, y_train, tokenizer)

# Create a validation dataset for the Trainer's evaluation steps
val_dataset = FEASDataset(X_val, y_val, tokenizer)

# Create the test dataset to use AFTER training is complete
test_dataset = FEASDataset(X_test, y_test, tokenizer)

### Model Initialization and Fine-Tuning

In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(TARGET_COLS),
    problem_type="multi_label_classification" 
)

# Custom metric function to compute Macro/Micro F1 during evaluation
def compute_metrics(p: EvalPrediction):
    logits = p.predictions
    labels = p.label_ids
    probs = 1 / (1 + np.exp(-logits))
    
    # Calculate PR-AUC (Average Precision) instead of F1
    # Note: average_precision_score handles the raw probabilities nicely
    macro_pr_auc = average_precision_score(labels, probs, average='macro')
    micro_pr_auc = average_precision_score(labels, probs, average='micro')
    
    return {'macro_pr_auc': macro_pr_auc, 'micro_pr_auc': micro_pr_auc}

# Define training arguments
training_args = TrainingArguments(
    output_dir='./distilbert_results',
    # Train for more epochs, but rely on early stopping to halt it
    num_train_epochs=15,                  
    per_device_train_batch_size=8,       # Small batches add helpful noise/regularization
    per_device_eval_batch_size=8,
    
    # Evaluate frequently (fraction of an epoch) instead of waiting for a full epoch
    eval_strategy="steps",
    eval_steps=15,                       # ~4 evaluations per epoch (500 rows / batch 8 = 62 steps)
    save_strategy="steps",
    save_steps=15,
    
    learning_rate=2e-5,                  # Keep the learning rate very conservative
    weight_decay=0.05,
    
    load_best_model_at_end=True,
    metric_for_best_model="macro_pr_auc",
    save_total_limit=2,

    dataloader_pin_memory = torch.cuda.is_available()
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)] 
)

# Start Fine-Tuning
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Macro Pr Auc,Micro Pr Auc
15,No log,0.440406,0.190587,0.268688
30,No log,0.332726,0.148887,0.218933
45,No log,0.301871,0.194713,0.269988
60,No log,0.293145,0.198492,0.266363
75,No log,0.289360,0.177596,0.262223
90,No log,0.286037,0.187381,0.262072
105,No log,0.284140,0.172813,0.272870
120,No log,0.282249,0.189782,0.293623


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=120, training_loss=0.3428724924723307, metrics={'train_runtime': 91.5987, 'train_samples_per_second': 48.636, 'train_steps_per_second': 6.223, 'total_flos': 62195661932544.0, 'train_loss': 0.3428724924723307, 'epoch': 3.1578947368421053})

In [27]:
predictions = trainer.predict(test_dataset)
logits = predictions.predictions

# Apply sigmoid to get probabilities for the 4 independent targets
y_prob_transformer = 1 / (1 + np.exp(-logits))

y_pred_transformer = np.zeros_like(y_prob_transformer, dtype=int)
for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] > 0.5).astype(int)

for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (DistilBERT) =====")
    print(classification_report(y_test[:, i], y_pred_transformer[:, i], digits=4, zero_division=0))

print("\nMacro F1:", f1_score(y_test, y_pred_transformer, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_test, y_pred_transformer, average='micro', zero_division=0))


===== feas_pa (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4727    1.0000    0.6420        26
           1     0.0000    0.0000    0.0000        29

    accuracy                         0.4727        55
   macro avg     0.2364    0.5000    0.3210        55
weighted avg     0.2235    0.4727    0.3035        55


===== feas_ka (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3273    1.0000    0.4932        18
           1     0.0000    0.0000    0.0000        37

    accuracy                         0.3273        55
   macro avg     0.1636    0.5000    0.2466        55
weighted avg     0.1071    0.3273    0.1614        55


===== feas_cr (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4182    1.0000    0.5897        23
           1     0.0000    0.0000    0.0000        32

    accuracy                         0.4182        55
   macro avg     0.2091   

In [28]:
# 1. Get probabilities on the VALIDATION set to tune thresholds
val_predictions = trainer.predict(val_dataset)
val_probs = 1 / (1 + np.exp(-val_predictions.predictions))

# 2. Tune thresholds
distilbert_thresholds = []
for i in range(len(TARGET_COLS)):
    t = tune_threshold(y_val[:, i], val_probs[:, i])
    distilbert_thresholds.append(t)
    
print("DistilBERT Tuned thresholds:", distilbert_thresholds)

# 3. Apply these thresholds to the TEST set
test_predictions = trainer.predict(test_dataset)
y_prob_transformer = 1 / (1 + np.exp(-test_predictions.predictions))

y_pred_transformer = np.zeros_like(y_prob_transformer, dtype=int)
for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] >= distilbert_thresholds[i]).astype(int)

DistilBERT Tuned thresholds: [np.float64(0.21530612244897956), np.float64(0.05), np.float64(0.05), np.float64(0.05)]


In [29]:
for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] >= distilbert_thresholds[i]).astype(int)

for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (DistilBERT) =====")
    print(classification_report(y_test[:, i], y_pred_transformer[:, i], digits=4, zero_division=0))

print("\nMacro F1:", f1_score(y_test, y_pred_transformer, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_test, y_pred_transformer, average='micro', zero_division=0))


===== feas_pa (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.5000    0.5769    0.5357        26
           1     0.5600    0.4828    0.5185        29

    accuracy                         0.5273        55
   macro avg     0.5300    0.5298    0.5271        55
weighted avg     0.5316    0.5273    0.5266        55


===== feas_ka (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        18
           1     0.6727    1.0000    0.8043        37

    accuracy                         0.6727        55
   macro avg     0.3364    0.5000    0.4022        55
weighted avg     0.4526    0.6727    0.5411        55


===== feas_cr (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        23
           1     0.5818    1.0000    0.7356        32

    accuracy                         0.5818        55
   macro avg     0.2909   

### Freezing Base Layers

In [30]:
model2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(TARGET_COLS),
    problem_type="multi_label_classification" 
)

# Freeze all base DistilBERT parameters
for param in model2.distilbert.parameters():
    param.requires_grad = False

training_args2 = TrainingArguments(
    output_dir='./distilbert_frozen_results',
    
    # Allow more epochs, but trust early stopping to catch the peak
    num_train_epochs=25,                  
    per_device_train_batch_size=8,       
    per_device_eval_batch_size=8,
    
    eval_strategy="steps",
    eval_steps=15,                       
    save_strategy="steps",
    save_steps=15,
    
    # Higher learning rate for the fresh classification head
    learning_rate=5e-4,                  
    
    # Strong regularization to combat the tiny dataset
    weight_decay=0.05,                   
    
    # Relying on PR-AUC for early stopping
    load_best_model_at_end=True,
    metric_for_best_model="macro_pr_auc",
    save_total_limit=2,                  

    dataloader_pin_memory = torch.cuda.is_available()
)

trainer2 = Trainer(
    model=model2,
    args=training_args2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)] 
)

# Start Fine-Tuning
trainer2.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Macro Pr Auc,Micro Pr Auc
15,No log,0.303166,0.099639,0.147477
30,No log,0.287777,0.139025,0.195465
45,No log,0.282014,0.164271,0.224013
60,No log,0.279543,0.193617,0.245605
75,No log,0.278160,0.208040,0.257304
90,No log,0.272976,0.217860,0.270967
105,No log,0.266147,0.255258,0.318552
120,No log,0.266090,0.281344,0.344088
135,No log,0.261630,0.300512,0.377705
150,No log,0.276452,0.307235,0.335304


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=315, training_loss=0.26289944118923614, metrics={'train_runtime': 330.5783, 'train_samples_per_second': 22.461, 'train_steps_per_second': 2.874, 'total_flos': 163205656018944.0, 'train_loss': 0.26289944118923614, 'epoch': 8.289473684210526})

In [31]:
predictions = trainer2.predict(test_dataset)
logits = predictions.predictions

# Apply sigmoid to get probabilities for the 4 independent targets
y_prob_transformer = 1 / (1 + np.exp(-logits))

y_pred_transformer = np.zeros_like(y_prob_transformer, dtype=int)
for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] > 0.5).astype(int)

for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (DistilBERT) =====")
    print(classification_report(y_test[:, i], y_pred_transformer[:, i], digits=4, zero_division=0))

print("\nMacro F1:", f1_score(y_test, y_pred_transformer, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_test, y_pred_transformer, average='micro', zero_division=0))


===== feas_pa (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4694    0.8846    0.6133        26
           1     0.5000    0.1034    0.1714        29

    accuracy                         0.4727        55
   macro avg     0.4847    0.4940    0.3924        55
weighted avg     0.4855    0.4727    0.3803        55


===== feas_ka (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3273    1.0000    0.4932        18
           1     0.0000    0.0000    0.0000        37

    accuracy                         0.3273        55
   macro avg     0.1636    0.5000    0.2466        55
weighted avg     0.1071    0.3273    0.1614        55


===== feas_cr (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4182    1.0000    0.5897        23
           1     0.0000    0.0000    0.0000        32

    accuracy                         0.4182        55
   macro avg     0.2091   

In [32]:
# 1. Get probabilities on the VALIDATION set to tune thresholds
val_predictions = trainer2.predict(val_dataset)
val_probs = 1 / (1 + np.exp(-val_predictions.predictions))

# 2. Tune thresholds
distilbert_thresholds = []
for i in range(len(TARGET_COLS)):
    t = tune_threshold(y_val[:, i], val_probs[:, i])
    distilbert_thresholds.append(t)
    
print("DistilBERT Tuned thresholds:", distilbert_thresholds)

# 3. Apply these thresholds to the TEST set
test_predictions = trainer2.predict(test_dataset)
y_prob_transformer = 1 / (1 + np.exp(-test_predictions.predictions))

y_pred_transformer = np.zeros_like(y_prob_transformer, dtype=int)

for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] >= distilbert_thresholds[i]).astype(int)

for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (DistilBERT) =====")
    print(classification_report(y_test[:, i], y_pred_transformer[:, i], digits=4, zero_division=0))

print("\nMacro F1:", f1_score(y_test, y_pred_transformer, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_test, y_pred_transformer, average='micro', zero_division=0))

DistilBERT Tuned thresholds: [np.float64(0.38061224489795914), np.float64(0.23367346938775507), np.float64(0.12346938775510204), 0.5]



===== feas_pa (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3939    0.5000    0.4407        26
           1     0.4091    0.3103    0.3529        29

    accuracy                         0.4000        55
   macro avg     0.4015    0.4052    0.3968        55
weighted avg     0.4019    0.4000    0.3944        55


===== feas_ka (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3250    0.7222    0.4483        18
           1     0.6667    0.2703    0.3846        37

    accuracy                         0.4182        55
   macro avg     0.4958    0.4962    0.4164        55
weighted avg     0.5548    0.4182    0.4054        55


===== feas_cr (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4444    0.8696    0.5882        23
           1     0.7000    0.2188    0.3333        32

    accuracy                         0.4909        55
   macro avg     0.5722   

### Custom Trainer with Weighted Loss

In [33]:
model3 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(TARGET_COLS),
    problem_type="multi_label_classification"
)

# Calculate positive weights based on the training data imbalance
# Formula for pos_weight: number_of_negatives / number_of_positives
pos_weights = []
for i in range(y_train.shape[1]):
    num_positives = y_train[:, i].sum()
    num_negatives = len(y_train) - num_positives
    # Add a small epsilon to avoid division by zero
    weight = num_negatives / (num_positives + 1e-6)
    pos_weights.append(weight)

pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float).to(model3.device)

# Subclass Trainer to use the weighted loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Apply the weighted BCE loss
        loss_fct = BCEWithLogitsLoss(pos_weight=pos_weights_tensor.to(logits.device))
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

training_args3 = TrainingArguments(
    output_dir='./distilbert_weighted_results',
    
    # Fewer epochs are needed when fine-tuning the whole model
    # Early stopping will still catch it if it converges before 15
    num_train_epochs=15,                  
    per_device_train_batch_size=8,       
    per_device_eval_batch_size=8,
    
    # Keep frequent evaluations to monitor the PR-AUC closely
    eval_strategy="steps",
    eval_steps=15,                       
    save_strategy="steps",
    save_steps=15,
    
    # Standard fine-tuning rate
    learning_rate=2e-5,                  
    
    # Keep the stronger L2 regularization to combat overfitting on the small dataset
    weight_decay=0.05,                   
    
    # Track the PR-AUC metric
    load_best_model_at_end=True,
    metric_for_best_model="macro_pr_auc",
    save_total_limit=2,                  

    dataloader_pin_memory=torch.cuda.is_available()
)

# Then use WeightedTrainer instead of Trainer
trainer3 = WeightedTrainer(
    model=model3,
    args=training_args3,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)] 
)

trainer3.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Macro Pr Auc,Micro Pr Auc
15,No log,1.330731,0.140742,0.143804
30,No log,1.401080,0.206400,0.207137
45,No log,1.532507,0.202313,0.248554
60,No log,1.548626,0.227562,0.177631
75,No log,1.612604,0.236466,0.261129
90,No log,1.693489,0.260419,0.275470
105,No log,1.631849,0.265852,0.279383
120,No log,1.665254,0.227870,0.300771
135,No log,1.627980,0.217990,0.298659
150,No log,1.633262,0.237476,0.253566


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=165, training_loss=1.2009747129498105, metrics={'train_runtime': 413.2254, 'train_samples_per_second': 10.781, 'train_steps_per_second': 1.379, 'total_flos': 85576991711232.0, 'train_loss': 1.2009747129498105, 'epoch': 4.342105263157895})

In [34]:
predictions = trainer3.predict(test_dataset)
logits = predictions.predictions

# Apply sigmoid to get probabilities for the 4 independent targets
y_prob_transformer = 1 / (1 + np.exp(-logits))

y_pred_transformer = np.zeros_like(y_prob_transformer, dtype=int)
for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] > 0.5).astype(int)

for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (DistilBERT) =====")
    print(classification_report(y_test[:, i], y_pred_transformer[:, i], digits=4, zero_division=0))

print("\nMacro F1:", f1_score(y_test, y_pred_transformer, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_test, y_pred_transformer, average='micro', zero_division=0))


===== feas_pa (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.5102    0.9615    0.6667        26
           1     0.8333    0.1724    0.2857        29

    accuracy                         0.5455        55
   macro avg     0.6718    0.5670    0.4762        55
weighted avg     0.6806    0.5455    0.4658        55


===== feas_ka (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3273    1.0000    0.4932        18
           1     0.0000    0.0000    0.0000        37

    accuracy                         0.3273        55
   macro avg     0.1636    0.5000    0.2466        55
weighted avg     0.1071    0.3273    0.1614        55


===== feas_cr (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4400    0.9565    0.6027        23
           1     0.8000    0.1250    0.2162        32

    accuracy                         0.4727        55
   macro avg     0.6200   

In [35]:
# 1. Get probabilities on the VALIDATION set to tune thresholds
val_predictions = trainer3.predict(val_dataset)
val_probs = 1 / (1 + np.exp(-val_predictions.predictions))

# 2. Tune thresholds
distilbert_thresholds = []
for i in range(len(TARGET_COLS)):
    t = tune_threshold(y_val[:, i], val_probs[:, i])
    distilbert_thresholds.append(t)
    
print("DistilBERT Tuned thresholds:", distilbert_thresholds)

# 3. Apply these thresholds to the TEST set
test_predictions = trainer3.predict(test_dataset)
y_prob_transformer = 1 / (1 + np.exp(-test_predictions.predictions))

y_pred_transformer = np.zeros_like(y_prob_transformer, dtype=int)

for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] >= distilbert_thresholds[i]).astype(int)

for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (DistilBERT) =====")
    print(classification_report(y_test[:, i], y_pred_transformer[:, i], digits=4, zero_division=0))

print("\nMacro F1:", f1_score(y_test, y_pred_transformer, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_test, y_pred_transformer, average='micro', zero_division=0))

DistilBERT Tuned thresholds: [np.float64(0.4724489795918367), np.float64(0.41734693877551016), np.float64(0.4540816326530612), np.float64(0.08673469387755102)]



===== feas_pa (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4444    0.4615    0.4528        26
           1     0.5000    0.4828    0.4912        29

    accuracy                         0.4727        55
   macro avg     0.4722    0.4721    0.4720        55
weighted avg     0.4737    0.4727    0.4731        55


===== feas_ka (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3333    0.7222    0.4561        18
           1     0.6875    0.2973    0.4151        37

    accuracy                         0.4364        55
   macro avg     0.5104    0.5098    0.4356        55
weighted avg     0.5716    0.4364    0.4285        55


===== feas_cr (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4400    0.4783    0.4583        23
           1     0.6000    0.5625    0.5806        32

    accuracy                         0.5273        55
   macro avg     0.5200   

### Weighted Loss + Freezing

In [36]:
model4 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(TARGET_COLS),
    problem_type="multi_label_classification"
)

for param in model4.distilbert.parameters():
    param.requires_grad = False

training_args4 = TrainingArguments(
    output_dir='./distilbert_weighted_frozen_results',
    
    # 1. Epochs and Batch Size
    # High epochs because we rely entirely on early stopping. 
    # Small batch size (8) acts as a natural regularizer for the small dataset.
    num_train_epochs=25,                  
    per_device_train_batch_size=8,       
    per_device_eval_batch_size=8,
    
    # 2. Evaluation and Saving Strategies
    # Evaluate frequently so the EarlyStoppingCallback can catch the exact peak
    eval_strategy="steps",
    eval_steps=15,                       
    save_strategy="steps",
    save_steps=15,
    
    # 3. Learning Rate and Regularization
    # The classification head is untrained, so it needs a much larger learning 
    # rate (e.g., 5e-4 or 1e-3) than standard fine-tuning (2e-5).
    learning_rate=5e-4,                  
    weight_decay=0.05,                   
    
    # 4. Metric Tracking
    load_best_model_at_end=True,
    metric_for_best_model="macro_pr_auc",
    greater_is_better=True,
    save_total_limit=2,                  

    dataloader_pin_memory=torch.cuda.is_available()
)

# Then use WeightedTrainer instead of Trainer
trainer4 = WeightedTrainer(
    model=model4,
    args=training_args4,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)] 
)

trainer4.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Macro Pr Auc,Micro Pr Auc
15,No log,1.655195,0.179156,0.211107
30,No log,1.797071,0.228006,0.223724
45,No log,1.930521,0.238207,0.213778
60,No log,2.053172,0.299259,0.349832
75,No log,1.772095,0.302005,0.355915
90,No log,1.878349,0.303985,0.296601
105,No log,1.699329,0.375387,0.363877
120,No log,1.706588,0.333790,0.265361
135,No log,1.632171,0.331798,0.334939
150,No log,1.850698,0.321766,0.275357


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=165, training_loss=1.5661055131392045, metrics={'train_runtime': 94.3671, 'train_samples_per_second': 78.682, 'train_steps_per_second': 10.067, 'total_flos': 85576991711232.0, 'train_loss': 1.5661055131392045, 'epoch': 4.342105263157895})

In [37]:
predictions = trainer4.predict(test_dataset)
logits = predictions.predictions

# Apply sigmoid to get probabilities for the 4 independent targets
y_prob_transformer = 1 / (1 + np.exp(-logits))

y_pred_transformer = np.zeros_like(y_prob_transformer, dtype=int)
for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] > 0.5).astype(int)

for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (DistilBERT) =====")
    print(classification_report(y_test[:, i], y_pred_transformer[:, i], digits=4, zero_division=0))

print("\nMacro F1:", f1_score(y_test, y_pred_transformer, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_test, y_pred_transformer, average='micro', zero_division=0))


===== feas_pa (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3750    0.3462    0.3600        26
           1     0.4516    0.4828    0.4667        29

    accuracy                         0.4182        55
   macro avg     0.4133    0.4145    0.4133        55
weighted avg     0.4154    0.4182    0.4162        55


===== feas_ka (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3182    0.7778    0.4516        18
           1     0.6364    0.1892    0.2917        37

    accuracy                         0.3818        55
   macro avg     0.4773    0.4835    0.3716        55
weighted avg     0.5322    0.3818    0.3440        55


===== feas_cr (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4545    0.8696    0.5970        23
           1     0.7273    0.2500    0.3721        32

    accuracy                         0.5091        55
   macro avg     0.5909   

In [38]:
# 1. Get probabilities on the VALIDATION set to tune thresholds
val_predictions = trainer4.predict(val_dataset)
val_probs = 1 / (1 + np.exp(-val_predictions.predictions))

# 2. Tune thresholds
distilbert_thresholds = []
for i in range(len(TARGET_COLS)):
    t = tune_threshold(y_val[:, i], val_probs[:, i])
    distilbert_thresholds.append(t)
    
print("DistilBERT Tuned thresholds:", distilbert_thresholds)

# 3. Apply these thresholds to the TEST set
test_predictions = trainer4.predict(test_dataset)
y_prob_transformer = 1 / (1 + np.exp(-test_predictions.predictions))

y_pred_transformer = np.zeros_like(y_prob_transformer, dtype=int)

for i in range(len(TARGET_COLS)):
    y_pred_transformer[:, i] = (y_prob_transformer[:, i] >= distilbert_thresholds[i]).astype(int)

for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (DistilBERT) =====")
    print(classification_report(y_test[:, i], y_pred_transformer[:, i], digits=4, zero_division=0))

print("\nMacro F1:", f1_score(y_test, y_pred_transformer, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_test, y_pred_transformer, average='micro', zero_division=0))

DistilBERT Tuned thresholds: [np.float64(0.5275510204081633), np.float64(0.5091836734693878), np.float64(0.5091836734693878), np.float64(0.06836734693877551)]



===== feas_pa (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3636    0.4615    0.4068        26
           1     0.3636    0.2759    0.3137        29

    accuracy                         0.3636        55
   macro avg     0.3636    0.3687    0.3603        55
weighted avg     0.3636    0.3636    0.3577        55


===== feas_ka (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.3125    0.8333    0.4545        18
           1     0.5714    0.1081    0.1818        37

    accuracy                         0.3455        55
   macro avg     0.4420    0.4707    0.3182        55
weighted avg     0.4867    0.3455    0.2711        55


===== feas_cr (DistilBERT) =====
              precision    recall  f1-score   support

           0     0.4348    0.8696    0.5797        23
           1     0.6667    0.1875    0.2927        32

    accuracy                         0.4727        55
   macro avg     0.5507   